# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Alpeshmore/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 158 (delta 64), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.94 MiB | 4.90 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [2]:
# ML-09 — Setup

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

# Reuse df and X if ML-08 variables already exist.
# Otherwise load the processed feature vector.

if "df" not in globals():

    possible_paths = [
        Path("data/processed/refresh_feature_vector.csv"),
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("/content/flyrank-ai/data/processed/refresh_feature_vector.csv"),
        Path("/content/flyrank-ai/data/raw/content_refresh_anonymized.csv"),
        Path("/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"),
        Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
    ]

    data_path = next((p for p in possible_paths if p.exists()), None)

    if data_path is None:
        raise FileNotFoundError(
            "Dataset not found. Run ML-05/ML-08 first or place the "
            "FlyRank dataset in data/raw/."
        )

    df = pd.read_csv(data_path)
    print("Loaded:", data_path)

# Build target if necessary
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

df["is_declining_label"] = df["is_declining_label"].astype(int)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Declining rate:", round(df["is_declining_label"].mean(), 4))

Loaded: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Rows: 30000
Clients: 32
Declining rate: 0.5421


## 1. Two paper findings + my methodology questions

### Finding 1 — The learned model can outperform the hand-written baseline

The research reports that the learned model can rank declining pages more effectively than the transparent rule-based baseline, measured using Precision@50.

The important methodology question is **where the label comes from**. In this dataset, `is_declining_label` is derived from `trend_direction`, so it is a proxy for observed decline rather than a directly measured business outcome such as a completed content refresh or recovered traffic.

The validation design therefore supports a claim about ranking pages associated with the observed decline label. It does **not** by itself prove that refreshing those pages will improve traffic or rankings.

### Finding 2 — Validation design affects how believable the model result is

The research workflow uses client-grouped validation so that pages from the same client are not placed in both training and test sets.

My methodology question is whether the reported split matches the deployment question. If the goal is to use the model on unseen clients, a client-holdout split is appropriate. If the goal is to rank additional pages for clients already represented in training, the question is different.

Overall, I would treat the result as evidence that the model has useful directional signal under the stated validation design, not as proof that the model will perform identically in every future setting.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic label audit supporting Section 1

print("Label source:")
print("is_declining_label = trend_direction == 'down'")

print("\nLabel counts:")
print(
    df["is_declining_label"]
    .value_counts()
    .rename(index={0: "not_declining", 1: "declining"})
)

print("\nDeclining percentage:")
print(
    round(df["is_declining_label"].mean() * 100, 2),
    "%"
)

print("\nClients:")
print(df["client_id"].nunique())

Label source:
is_declining_label = trend_direction == 'down'

Label counts:
is_declining_label
declining        16262
not_declining    13738
Name: count, dtype: int64

Declining percentage:
54.21 %

Clients:
32


## 2. My model under an honest split

For the **before** result, I use a standard random row split. This can place pages from the same client in both training and test data.

For the **after** result, I hold out complete clients. No client appears in both training and test data.

I keep the model, features, target, random seed, and Precision@50 metric the same. This means the main change is the validation design.

If the score falls under the grouped split, I will report the lower number as the more honest estimate for generalization to unseen clients.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Build the final feature matrix
# ---------------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [
    c for c in numeric_features
    if c in df.columns
]

categorical_features = [
    c for c in categorical_features
    if c in df.columns
]

numeric_frame = (
    df[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

categorical_frame = (
    df[categorical_features]
    .fillna("unknown")
    .astype(str)
)

encoded_frame = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dtype=float
)

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_frame.reset_index(drop=True)
    ],
    axis=1
)

y = df["is_declining_label"].astype(int).reset_index(drop=True)

print("Feature matrix:", X.shape)
print("Target:", y.shape)

Feature matrix: (30000, 48)
Target: (30000,)


In [5]:
# ---------------------------------------------------------
# Safety check — no obvious leakage fields
# ---------------------------------------------------------

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
]

leakage_columns_present = [
    c for c in forbidden_features
    if c in X.columns
]

print("Forbidden fields found in X:", leakage_columns_present)

assert len(leakage_columns_present) == 0

print("Feature leakage check: PASS")

Forbidden fields found in X: []
Feature leakage check: PASS


In [6]:
# ---------------------------------------------------------
# BEFORE: random row split
# ---------------------------------------------------------

indices = np.arange(len(df))

train_idx_random, test_idx_random = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

rf_before = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_before.fit(
    X.iloc[train_idx_random],
    y.iloc[train_idx_random]
)

before_scores = rf_before.predict_proba(
    X.iloc[test_idx_random]
)[:, 1]

y_before = y.iloc[test_idx_random].to_numpy()

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    order = np.argsort(-scores, kind="stable")[:k]
    return float(np.asarray(y_true)[order].mean())

before_p20 = precision_at_k(y_before, before_scores, 20)
before_p50 = precision_at_k(y_before, before_scores, 50)
before_p100 = precision_at_k(y_before, before_scores, 100)

before_auc = roc_auc_score(
    y_before,
    before_scores
)

print("BEFORE — random row split")
print("Precision@20 :", round(before_p20, 3))
print("Precision@50 :", round(before_p50, 3))
print("Precision@100:", round(before_p100, 3))
print("ROC-AUC      :", round(before_auc, 3))

BEFORE — random row split
Precision@20 : 0.85
Precision@50 : 0.9
Precision@100: 0.89
ROC-AUC      : 0.757


In [7]:
# ---------------------------------------------------------
# AFTER: client-grouped split
# ---------------------------------------------------------

clients = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = clients.unique()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:test_client_count]
)

group_test_mask = clients.isin(test_clients).to_numpy()

train_idx_grouped = indices[~group_test_mask]
test_idx_grouped = indices[group_test_mask]

rf_after = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_after.fit(
    X.iloc[train_idx_grouped],
    y.iloc[train_idx_grouped]
)

after_scores = rf_after.predict_proba(
    X.iloc[test_idx_grouped]
)[:, 1]

y_after = y.iloc[test_idx_grouped].to_numpy()

after_p20 = precision_at_k(
    y_after,
    after_scores,
    20
)

after_p50 = precision_at_k(
    y_after,
    after_scores,
    50
)

after_p100 = precision_at_k(
    y_after,
    after_scores,
    100
)

after_auc = roc_auc_score(
    y_after,
    after_scores
)

print("AFTER — client-grouped split")
print("Precision@20 :", round(after_p20, 3))
print("Precision@50 :", round(after_p50, 3))
print("Precision@100:", round(after_p100, 3))
print("ROC-AUC      :", round(after_auc, 3))

AFTER — client-grouped split
Precision@20 : 0.7
Precision@50 : 0.74
Precision@100: 0.71
ROC-AUC      : 0.744


In [8]:
# ---------------------------------------------------------
# BEFORE vs AFTER table
# ---------------------------------------------------------

validation_comparison = pd.DataFrame([
    {
        "Validation": "Before — random rows",
        "Test rows": len(test_idx_random),
        "Test clients": df.iloc[test_idx_random]["client_id"].nunique(),
        "Precision@20": before_p20,
        "Precision@50": before_p50,
        "Precision@100": before_p100,
        "ROC-AUC": before_auc,
    },
    {
        "Validation": "After — client grouped",
        "Test rows": len(test_idx_grouped),
        "Test clients": df.iloc[test_idx_grouped]["client_id"].nunique(),
        "Precision@20": after_p20,
        "Precision@50": after_p50,
        "Precision@100": after_p100,
        "ROC-AUC": after_auc,
    }
])

print(
    validation_comparison
    .round(3)
    .to_string(index=False)
)

print("\nPrecision@50 change:")
print(
    round(after_p50 - before_p50, 3)
)

            Validation  Test rows  Test clients  Precision@20  Precision@50  Precision@100  ROC-AUC
  Before — random rows       6000            31          0.85          0.90           0.89    0.757
After — client grouped       2325             6          0.70          0.74           0.71    0.744

Precision@50 change:
-0.16


In [9]:
# ---------------------------------------------------------
# Confirm there is NO client overlap after grouped split
# ---------------------------------------------------------

train_clients = set(
    df.iloc[train_idx_grouped]["client_id"]
    .astype(str)
)

test_clients_actual = set(
    df.iloc[test_idx_grouped]["client_id"]
    .astype(str)
)

overlap = train_clients.intersection(
    test_clients_actual
)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("Grouped split check: PASS")

Client overlap: 0
Grouped split check: PASS


## 3. Leakage audit

I checked the final feature set for three main leakage risks.

First, I excluded fields directly used to create the label, especially `trend_direction` and `trend_pct`.

Second, I excluded identifiers such as `content_id` and `client_id` from the feature matrix. `client_id` is used only to create the grouped validation split.

Third, I checked for decision-derived fields such as an existing priority, refresh recommendation, or action label. These would allow the model to learn a decision that was already made instead of learning from independent evidence.

The final feature matrix therefore contains search, traffic, content, freshness, and engagement signals, while the label and grouping identifiers remain outside the model features.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Leakage audit — explicit checks
# ---------------------------------------------------------

print("LEAKAGE AUDIT")
print("=" * 50)

# 1. Direct label leakage
label_derived = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

for col in label_derived:
    print(
        f"{col:25s}:",
        "LEAKAGE RISK" if col in X.columns else "excluded"
    )

# 2. IDs / grouping fields
grouping_fields = [
    "content_id",
    "client_id",
]

for col in grouping_fields:
    print(
        f"{col:25s}:",
        "GROUPING ONLY" if col in df.columns else "not present"
    )

# 3. Common decision-derived fields
decision_fields = [
    "priority_score",
    "health_score",
    "action_type",
    "refresh_tier",
    "refresh_flag",
    "measurable_opportunity",
]

for col in decision_fields:
    if col in df.columns:
        print(
            f"{col:25s}: decision-derived field — excluded"
        )

# Final hard check
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "is_declining_label" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("\nFINAL LEAKAGE CHECK: PASS")

LEAKAGE AUDIT
trend_direction          : excluded
trend_pct                : excluded
is_declining_label       : excluded
content_id               : GROUPING ONLY
client_id                : GROUPING ONLY

FINAL LEAKAGE CHECK: PASS


In [11]:
# ---------------------------------------------------------
# Check whether train/test client overlap would exist
# under the BEFORE random split.
# ---------------------------------------------------------

random_train_clients = set(
    df.iloc[train_idx_random]["client_id"]
    .astype(str)
)

random_test_clients = set(
    df.iloc[test_idx_random]["client_id"]
    .astype(str)
)

random_overlap = random_train_clients.intersection(
    random_test_clients
)

print("Random split client overlap:", len(random_overlap))

if len(random_overlap) > 0:
    print(
        "This confirms that the random split contains clients "
        "represented in both training and test data."
    )
else:
    print(
        "No overlap happened in this random split."
    )

Random split client overlap: 31
This confirms that the random split contains clients represented in both training and test data.


## 4. Claim rewrite

### My original bold claim

“My Random Forest predicts which pages need to be refreshed and performs much better than the rule-based approach.”

### Safer research claim

“On the evaluated anonymized dataset, the Random Forest showed higher observed Precision@50 than the hand-written baseline under the tested validation design. The result is directional decision-support for prioritizing pages associated with observed decline; it does not establish that a refresh will cause improved search performance or generalize unchanged to every future client.”


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Generate the exact numbers for the final claim
# ---------------------------------------------------------

score_change = after_p50 - before_p50

if before_p50 > 0:
    relative_change = (
        (after_p50 - before_p50) / before_p50
    )
else:
    relative_change = np.nan

print("FINAL VALIDATION SUMMARY")
print("=" * 50)

print(
    f"Random split Precision@50: {before_p50:.3f}"
)

print(
    f"Client-grouped Precision@50: {after_p50:.3f}"
)

print(
    f"Change: {score_change:+.3f}"
)

if not np.isnan(relative_change):
    print(
        f"Relative change: {relative_change:+.1%}"
    )

print(
    "\nUse the client-grouped number as the more honest "
    "estimate when your intended use is unseen clients."
)

FINAL VALIDATION SUMMARY
Random split Precision@50: 0.900
Client-grouped Precision@50: 0.740
Change: -0.160
Relative change: -17.8%

Use the client-grouped number as the more honest estimate when your intended use is unseen clients.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.